# Geospatial Visualization with Python

This notebook shows four visualization patterns: **raster data**, **vector data**, **combining raster and vector** layers, and **3D objects**. We use [leafmap](https://leafmap.org/) for interactive 2D and 3D maps.

In [ ]:
# Optional: Install packages if needed
# %pip install leafmap geopandas rasterio

import leafmap
import geopandas as gpd
from pathlib import Path
import numpy as np

## Example 1: Visualizing Raster Data

Raster visualization displays pixel values as colors. For single-band data (e.g., elevation), we use a colormap. For multi-band RGB imagery, we display bands 1–3 as red, green, and blue.

In [ ]:
# Use a remote Cloud Optimized GeoTIFF (COG) for raster display
cog_url = "https://github.com/opengeos/data/releases/download/raster/Libya-2023-07-01.tif"

In [ ]:
# Visualize the raster with leafmap
m = leafmap.Map()
m.add_cog_layer(cog_url, name="Satellite imagery (Libya)", bands=["b1", "b2", "b3"])
m

**Interpretation**: `add_cog_layer` streams the raster from a URL without downloading the full file. Bands `b1`, `b2`, `b3` are displayed as RGB. The map auto-centers on the COG extent.

## Example 2: Visualizing Vector Data

Vector visualization draws points, lines, and polygons on a map. We can color features by an attribute (e.g., population, land use) or use a single style for all features.

In [ ]:
from shapely.geometry import Point, Polygon

# Create sample vector data in WGS84 (lat/lon) for leafmap display
points = [
    Point(-122.42, 37.78), Point(-122.40, 37.79), Point(-122.38, 37.78),
    Point(-122.40, 37.80), Point(-122.36, 37.79)
]
polygons = [
    Polygon([(-122.44, 37.76), (-122.42, 37.76), (-122.42, 37.78), (-122.44, 37.78), (-122.44, 37.76)]),
    Polygon([(-122.40, 37.78), (-122.38, 37.78), (-122.38, 37.80), (-122.40, 37.80), (-122.40, 37.78)]),
]
gdf_points = gpd.GeoDataFrame(
    {"id": range(len(points)), "value": [10, 20, 15, 25, 30]},
    geometry=points, crs="EPSG:4326"
)
gdf_polygons = gpd.GeoDataFrame(
    {"id": range(len(polygons)), "type": ["A", "B"]},
    geometry=polygons, crs="EPSG:4326"
)

In [ ]:
# Visualize vector data with leafmap
m = leafmap.Map(center=[37.78, -122.40], zoom=12)
m.add_vector(gdf_polygons, layer_name="Polygons", style={"fillColor": "lightblue", "color": "darkblue", "fillOpacity": 0.6})
m.add_vector(gdf_points, layer_name="Points", style={"color": "red", "weight": 3})
m

**Interpretation**: Leafmap displays polygons and points as interactive layers. Use the layer control to toggle visibility. This pattern is useful for choropleth maps and thematic overlays.

## Example 3: Combining Raster and Vector Data

Overlaying vector features on a raster base map is a common workflow: e.g., building footprints on satellite imagery, or administrative boundaries on a land cover map. Both layers must share the same CRS.

In [ ]:
# Use the same vector data (already in WGS84)
gdf_polygons_match = gdf_polygons
gdf_points_match = gdf_points

In [ ]:
# Combine raster and vector with leafmap
m = leafmap.Map(center=[37.78, -122.40], zoom=12)
m.add_basemap("Esri.WorldImagery")  # Satellite basemap as raster backdrop
m.add_vector(gdf_polygons_match, layer_name="Polygons", style={"color": "red", "weight": 2, "fillOpacity": 0})
m.add_vector(gdf_points_match, layer_name="Points", style={"color": "yellow", "weight": 2})
m

**Interpretation**: The satellite basemap provides the raster backdrop; vector layers are drawn on top. Red polygon outlines and yellow points show how features align with the imagery. In real applications, you might overlay building footprints on aerial imagery or roads on a land cover map.

## Example 4: Displaying 3D Objects

Leafmap's **pydeck** backend supports 3D visualization. Polygons can be **extruded** (given height) based on an attribute, and points can be shown as 3D columns. Use `import leafmap.deck as leafmap` and set `extruded=True` with `get_elevation` to map an attribute to height. Press **Ctrl + left mouse** to rotate the 3D view.

In [ ]:
import leafmap.deck as leafmap

# 3D map with extruded polygons (US states by land area)
m = leafmap.Map(
    center=(40, -100),
    zoom=3,
    initial_view_state={"pitch": 45, "bearing": 10},
)
url = "https://github.com/giswqs/streamlit-geospatial/raw/master/data/us_states.geojson"
m.add_vector(
    url,
    random_color_column="STATEFP",
    extruded=True,
    get_elevation="ALAND",
    elevation_scale=0.000001,
)
m

**Interpretation**: Each state polygon is extruded vertically by its land area (`ALAND`). The 3D view reveals spatial patterns that are harder to see in 2D. Use `elevation_scale` to adjust the height exaggeration.

In [ ]:
# 3D column layer: points with height (funding amounts as column height)
import leafmap.deck as leafmap
m = leafmap.Map(center=(40, -100), zoom=3)
gdf_3d = gpd.read_file(
    "https://data.source.coop/cboettig/conservation-policy/Inflation_Reduction_Act_Projects.geojson"
)
# Keep only rows with valid funding for 3D extrusion
gdf_3d = gdf_3d[gdf_3d["FUNDING_NUMERIC"].notna()]
m.add_vector(
    gdf_3d,
    layer_type="ColumnLayer",
    get_position=["LONGITUDE", "LATITUDE"],
    get_elevation="FUNDING_NUMERIC",
    get_fill_color=[255, 200, 0, 180],
    elevation_scale=0.01,
    radius=15000,
    pickable=True,
)
m

## Summary

| Example | Tools | Use case |
|---------|-------|----------|
| Raster | `leafmap.add_cog_layer` | Satellite imagery, remote COG streaming |
| Vector | `leafmap.add_vector` | Points, lines, polygons with attributes |
| Combined | `add_basemap` + `add_vector` | Raster backdrop with vector overlay |
| 3D | `leafmap.deck` + `extruded` / `ColumnLayer` | Extruded polygons, 3D columns by attribute |